# 第94章 ROC、PR曲线与决策阈值

使用 ROC、PR 曲线和阈值表评估概率排序，并根据成本选择决策阈值。


## 先解决一个小问题

围绕“ROC、PR曲线与决策阈值”完成一个可验证的小型建模实验：先明确输入和目标，再比较方法带来的变化。使用 ROC、PR 曲线和阈值表评估概率排序，并根据成本选择决策阈值。


## 这章为什么先学

这是“机器学习”建模主线中的第 94 章，重点放在“ROC、PR曲线与决策阈值”对应的一个具体决策，而不是重复完整流程。


## 开始前确认

- 能够使用 pandas 读取、筛选和汇总数据
- 理解训练集、测试集和基本统计指标
- 本章会进一步练习：计算 ROC-AUC 与 PR-AUC、理解 TPR、FPR 和 Precision、生成阈值性能表


## 做完要留下什么

完成一份围绕“ROC、PR曲线与决策阈值”的可运行实验：包含数据准备、方法执行、指标或图表证据，以及一句有边界的结论。


## 运行规则

代码单元格按依赖顺序执行；需要复现结果时从上到下运行，并保留输入、计算和输出。


## 本章要会

- 计算 ROC-AUC 与 PR-AUC
- 理解 TPR、FPR 和 Precision
- 生成阈值性能表
- 按错误成本选择阈值


## 核心概念

- TPR=TP/(TP+FN)
- FPR=FP/(FP+TN)
- PR-AUC 更关注正类稀少任务
- 阈值选择不能依赖最终测试集


## 示例 1：数据与问题定义

先明确样本、特征、目标和验证方式，再训练模型。


In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import numpy as np

data = load_breast_cancer(as_frame=True); X, y=data.data, data.target
X_train, X_test, y_train, y_test=train_test_split(X, y, stratify=y, random_state=94)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train, y_train)
prob = model.predict_proba(X_test)[:,1]


## 示例 2：模型、公式与诊断

把核心数学量映射到 sklearn 输出，并检查泛化表现。


In [ ]:
import pandas as pd

print('ROC-AUC/PR-AUC:', round(roc_auc_score(y_test, prob),3), round(average_precision_score(y_test, prob),3))
rows = []
for t in np.arange(.1,1,.1):
    p=prob>=t; rows.append([t, precision_score(y_test, p), recall_score(y_test, p), p.mean()])
thresholds = pd.DataFrame(rows, columns=['threshold', 'precision', 'recall', 'positive_rate'])
display(thresholds.round(3))


## 建模流程提醒

1. **定义问题**：写清楚样本粒度、预测时点、目标变量和业务代价。
2. **建立基线**：先用均值、规则或 Dummy 模型得到最低可接受结果。
3. **准备数据**：只用预测时点可获得的信息，避免目标泄漏和时间穿越。
4. **训练与验证**：在训练/验证数据上选择方案，测试集只用于最终估计泛化表现。
5. **评价与解释**：同时看总体指标、错误切片和结果边界，不能只报一个分数。


## 常见误区

- AUC 高就忽略阈值表现
- 在测试集上挑阈值
- 正类比例变化后仍沿用旧 PR 基线
- 没有考虑复核容量


## 综合练习

1. 修改一个关键参数并重新运行
2. 记录指标变化并解释原因
3. 检查结论是否依赖测试集或隐藏泄漏

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“修改一个关键参数并重新运行”。
2. **独立完成**：不复制示例代码，完成“记录指标变化并解释原因”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“检查结论是否依赖测试集或隐藏泄漏”，用一两句话说明你修改了什么。

### 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


In [ ]:
eligible=thresholds.query('recall>=0.95')
selected_threshold = float(eligible.loc[eligible.precision.idxmax(), 'threshold'])
print('满足召回约束的高精确率阈值:', selected_threshold)

# 自检
assert 0<selected_threshold<1


## 本章小结

使用 ROC、PR 曲线和阈值表评估概率排序，并根据成本选择决策阈值。


### 你已经掌握

- 计算 ROC-AUC 与 PR-AUC
- 理解 TPR、FPR 和 Precision
- 生成阈值性能表
- 按错误成本选择阈值


### 验收标准

- 输入、计算和输出单元格完整。
- 关键变量类型、形状或数值可核对。
- 结论引用输出证据，并注明适用范围。


### 关键知识速查

| 知识点 | 作用与提醒 | 关键写法 |
| --- | --- | --- |
| 数据与问题定义 | 先明确样本、特征、目标和验证方式，再训练模型。 | `model.predict_proba()`、`.fit()` |
| 模型、公式与诊断 | 把核心数学量映射到 sklearn 输出，并检查泛化表现。 | `np.arange()`、`rows.append()`、`p.mean()`、`pd.DataFrame()` |


### 需要注意

- AUC 高就忽略阈值表现
- 在测试集上挑阈值
- 正类比例变化后仍沿用旧 PR 基线
- 没有考虑复核容量


### 完成检查

- [ ] 能够计算 ROC-AUC 与 PR-AUC
- [ ] 能够理解 TPR、FPR 和 Precision
- [ ] 能够生成阈值性能表
- [ ] 能够按错误成本选择阈值


### 排错顺序

1. 从上到下重新运行依赖单元格。
2. 检查变量类型、列名、形状和缺失值。
3. 缩小输入范围，定位产生错误的最小步骤。
4. 修复后重新运行完整流程。
